# PJM Merit Order Stack & Marginal Abatement Cost Curve
---

This notebook constructs a **merit-order supply stack** and a **marginal abatement cost curve (MACC)** for the PJM Interconnection using plant-level data from `PJM_Stack_Model_v1_2026_mar_10.xlsx`.

**Methodology** (adapted from [Merit Order & MACC in Python](https://medium.com/data-science/merit-order-and-marginal-abatement-cost-curve-in-python-fe9f77358777)):

1. **Merit Order**: Sort all generating units by marginal production cost ($/MWh) ascending. Plot cumulative capacity (MW) on the x-axis vs. marginal cost on the y-axis as a step function. The market clearing price for any load level is where the demand curve intersects the supply stack.

2. **MACC**: For each fossil unit, compute CO2 emission intensity (tons CO2/MWh). Compare each unit against a reference clean technology (efficient gas CC) to calculate the abatement cost ($/ton CO2) for fuel switching. Sort by abatement cost and plot cumulative abatement potential.

**Data sources:**
- `Stack Model` sheet — 4,500+ PJM generating units with capacity, heat rate, fuel hub, fuel price, VOM, and must-run status
- `Assumptions` sheet — fuel prices, CO2 emissions factors, RGGI carbon price

In [3]:
import pandas as pd
import numpy as np

In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

FILE = '../../.excel/PJM_Stack_Model_v1_2026_mar_10.xlsx'

## 1. Load Assumptions

Parse fuel prices, emissions factors, and carbon prices from the `Assumptions` sheet. These drive the marginal cost calculation for every thermal unit in PJM.

In [7]:
# --- Read Assumptions sheet (key-value layout, not tabular) ---
raw = pd.read_excel(FILE, sheet_name='Assumptions', header=None, usecols=[0, 1])
raw.columns = ['param', 'value']
raw = raw.dropna(subset=['value'])
params = dict(zip(raw['param'], raw['value']))

# --- Fuel prices ($/MMBtu) ---
fuel_prices = {
    'Columbia TCO Pool': params.get('Columbia TCO Pool', 2.60),
    'Dominion South Pt': params.get('Dominion South Pt', 2.45),
    'Northern Ventura':  params.get('Northern Ventura', 2.65),
    'Tetco M2 (deliveries)': params.get('Tetco M2 (deliveries)', 2.70),
    'Tetco M3':          params.get('Tetco M3', 2.85),
    'Transco Leidy':     params.get('Transco Leidy', 2.55),
    'Transco Z5 non-WGL': params.get('Transco Z5 non-WGL', 2.75),
    'Transco Z6':        params.get('Transco Z6', 2.90),
    'Big Sandy (Barge)': params.get('Big Sandy (Barge)', 2.50),
    'Central App (CSX)': params.get('Central App (CSX)', 2.65),
    'Northern App (Penn Railcar)': params.get('Northern App (Penn Railcar)', 2.80),
    'PRB 8800':          params.get('PRB 8800', 1.20),
    'Uinta Basin':       params.get('Uinta Basin', 1.90),
    'Gulf Coast No 6 Distillate': params.get('Gulf Coast No 6 Distillate', 9.50),
    'NY No 2 Distillate': params.get('NY No 2 Distillate', 16.00),
    'NY No 54 Jet Fuel': params.get('NY No 54 Jet Fuel', 17.00),
    'Uranium 308':       params.get('Uranium 308', 0.70),
}

# --- Emissions factors (tons CO2 / MMBtu) ---
# Try exact key match first, fall back to partial match
def find_param(search_term, default):
    for k, v in params.items():
        if isinstance(k, str) and search_term in k:
            return v
    return default

ef_gas  = find_param('Gas (tons CO2', 0.053)
ef_coal = find_param('Coal (tons CO2', 0.097)
ef_oil  = find_param('Oil (tons CO2', 0.075)
emissions_factors = {'Gas': ef_gas, 'Coal': ef_coal, 'Oil': ef_oil}

# --- Carbon & SO2 prices ---
rggi_price = find_param('RGGI', 15.0)

print('=== Fuel Prices ($/MMBtu) ===')
for k, v in fuel_prices.items():
    print(f'  {k:30s}  ${v:.2f}')
print(f'\n=== Emissions Factors (tons CO2/MMBtu) ===')
print(f'  Gas:  {ef_gas}   Coal: {ef_coal}   Oil: {ef_oil}')
print(f'\n=== Carbon Price ===')
print(f'  RGGI: ${rggi_price:.2f}/ton CO2')

=== Fuel Prices ($/MMBtu) ===
  Columbia TCO Pool               $2.60
  Dominion South Pt               $2.45
  Northern Ventura                $2.65
  Tetco M2 (deliveries)           $2.70
  Tetco M3                        $2.85
  Transco Leidy                   $2.55
  Transco Z5 non-WGL              $2.75
  Transco Z6                      $2.90
  Big Sandy (Barge)               $2.50
  Central App (CSX)               $2.65
  Northern App (Penn Railcar)     $2.80
  PRB 8800                        $1.20
  Uinta Basin                     $1.90
  Gulf Coast No 6 Distillate      $9.50
  NY No 2 Distillate              $16.00
  NY No 54 Jet Fuel               $17.00
  Uranium 308                     $0.70

=== Emissions Factors (tons CO2/MMBtu) ===
  Gas:  0.053   Coal: 0.097   Oil: 0.075

=== Carbon Price ===
  RGGI: $15.00/ton CO2


## 2. Load Stack Model Data

Read the `Stack Model` sheet which contains ~4,500 PJM generating units already organized by power hub with pre-linked fuel prices.

> **Important correction**: The workbook's fuel cost formula divides by 1,000 erroneously
> (`=HR*Price/1000`). Since heat rates are stored in MMBTU/MWh (≡ kBTU/kWh), the correct
> formula is simply `Fuel Cost = HR × Price`. We recompute all costs from scratch.

In [8]:
# --- Read Stack Model (header on row 3, 0-indexed=2) ---
df = pd.read_excel(FILE, sheet_name='Stack Model', header=2)

# --- Clean column names ---
col_map = {
    df.columns[0]:  'power_hub',
    df.columns[1]:  'plant_name',
    df.columns[2]:  'fuel_category',
    df.columns[3]:  'unit_type',
    df.columns[4]:  'summer_cap_mw',
    df.columns[5]:  'min_load_mw',
    df.columns[6]:  'heat_rate',       # MMBTU/MWh (labeled BTU/kWh in workbook)
    df.columns[7]:  'fuel_hub',
    df.columns[8]:  'fuel_price',      # $/MMBtu, linked from Assumptions
    df.columns[9]:  'fuel_cost_orig',  # workbook formula (has /1000 error)
    df.columns[10]: 'vom',             # $/MWh
    df.columns[11]: 'carbon_cost_orig',
    df.columns[12]: 'marginal_cost_orig',
    df.columns[13]: 'cum_cap_hub',
    df.columns[14]: 'must_run',
    df.columns[15]: 'on_off',
}
# Only rename columns that exist
df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})

# --- Drop section-header rows (merged cells with no plant name) ---
df = df.dropna(subset=['plant_name'])

# --- Filter to active units ---
df = df[df['on_off'] == 1].copy()

# --- Coerce numeric columns ---
for col in ['summer_cap_mw', 'min_load_mw', 'heat_rate', 'fuel_price', 'vom']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print(f'Loaded {len(df):,} active units  |  Total capacity: {df["summer_cap_mw"].sum():,.0f} MW')
print(f'Fuel categories: {sorted(df["fuel_category"].unique())}')
print(f'Power hubs: {df["power_hub"].nunique()}')
print(f'Must-run: {(df["must_run"]=="Yes").sum():,} units  |  Dispatchable: {(df["must_run"]=="No").sum():,} units')

Loaded 4,593 active units  |  Total capacity: 217,474 MW
Fuel categories: ['Biomass', 'Coal', 'Gas CC', 'Gas CT/ST', 'Hydro', 'Nuclear', 'Oil', 'Other', 'Solar', 'Storage', 'Wind']
Power hubs: 17
Must-run: 3,294 units  |  Dispatchable: 1,299 units


## 3. Data Quality Checks

In [9]:
# --- Nulls ---
null_counts = df[['power_hub','plant_name','fuel_category','summer_cap_mw','heat_rate','vom']].isnull().sum()
print('=== Null Counts (key columns) ===')
print(null_counts[null_counts > 0] if null_counts.any() else '  No nulls in key columns')

# --- Zero or negative capacity ---
bad_cap = df[df['summer_cap_mw'] <= 0]
print(f'\nUnits with zero/negative capacity: {len(bad_cap)}')

# --- Heat rate sanity (thermal units should have HR > 0) ---
thermal_fuels = ['Coal', 'Gas CC', 'Gas CT/ST', 'Oil']
thermal = df[df['fuel_category'].isin(thermal_fuels)]
zero_hr = thermal[thermal['heat_rate'] == 0]
print(f'\nThermal units with zero heat rate: {len(zero_hr)} / {len(thermal)}')
if len(zero_hr) > 0:
    print('  (These are likely must-run units modeled without fuel cost)')
    print(f'  Must-run: {(zero_hr["must_run"]=="Yes").sum()}  |  Dispatchable: {(zero_hr["must_run"]=="No").sum()}')
    print(f'  Capacity: {zero_hr["summer_cap_mw"].sum():,.0f} MW')

# --- Fuel price coverage ---
missing_fp = df[(df['fuel_category'].isin(thermal_fuels)) & (df['heat_rate'] > 0) & (df['fuel_price'] == 0)]
print(f'\nThermal units (HR>0) with missing fuel price: {len(missing_fp)}')

# --- Duplicates (same plant, same hub, same capacity) ---
dup_cols = ['power_hub', 'plant_name', 'fuel_category', 'summer_cap_mw', 'heat_rate']
dupes = df.duplicated(subset=dup_cols, keep=False)
print(f'\nPotential duplicates (same plant/hub/cap/HR): {dupes.sum()} rows')
print('  (Multiple units at the same plant with identical specs is expected for multi-unit stations)')

=== Null Counts (key columns) ===
  No nulls in key columns

Units with zero/negative capacity: 37

Thermal units with zero heat rate: 416 / 1715
  (These are likely must-run units modeled without fuel cost)
  Must-run: 416  |  Dispatchable: 0
  Capacity: 7,245 MW

Thermal units (HR>0) with missing fuel price: 0

Potential duplicates (same plant/hub/cap/HR): 1879 rows
  (Multiple units at the same plant with identical specs is expected for multi-unit stations)


## 4. Compute Corrected Marginal Costs

| Component | Formula | Notes |
|-----------|---------|-------|
| **Fuel Cost** | Heat Rate (MMBTU/MWh) × Fuel Price ($/MMBTU) | Workbook erroneously divides by 1,000 |
| **CO2 Rate** | Heat Rate × Emissions Factor (by fuel type) | tons CO2/MWh |
| **Carbon Cost** | CO2 Rate × RGGI Price | Only for units in RGGI states (NJ, DE, MD, VA) |
| **Marginal Cost** | Fuel Cost + VOM + Carbon Cost | $/MWh |

In [10]:
# --- Map fuel category to base fuel type for emissions ---
def fuel_base(cat):
    if cat in ('Gas CC', 'Gas CT/ST'):
        return 'Gas'
    elif cat == 'Coal':
        return 'Coal'
    elif cat == 'Oil':
        return 'Oil'
    return None

df['fuel_base'] = df['fuel_category'].apply(fuel_base)

# --- Corrected fuel cost (HR x Price, no /1000) ---
df['fuel_cost'] = df['heat_rate'] * df['fuel_price']

# --- CO2 emissions intensity (tons CO2/MWh) ---
df['co2_rate'] = df['heat_rate'] * df['fuel_base'].map(emissions_factors).fillna(0)

# --- Carbon cost (RGGI applies to NJ, DE, MD, VA hubs) ---
rggi_hubs = ['PJM AE', 'PJM BGE', 'PJM DPL', 'PJM Dominion',
             'PJM JCPL', 'PJM PEPCO', 'PJM PSEG']
df['in_rggi'] = df['power_hub'].isin(rggi_hubs)
df['carbon_cost'] = np.where(
    df['in_rggi'] & (df['co2_rate'] > 0),
    df['co2_rate'] * rggi_price,
    0.0
)

# --- Total marginal cost ---
df['marginal_cost'] = df['fuel_cost'] + df['vom'] + df['carbon_cost']

# --- Verify with examples ---
print('=== Sample Corrected Marginal Costs (thermal units, sorted) ===')
sample_cols = ['plant_name', 'fuel_category', 'summer_cap_mw', 'heat_rate',
               'fuel_price', 'fuel_cost', 'vom', 'carbon_cost', 'marginal_cost']
display_df = df[df['heat_rate'] > 0].sort_values('marginal_cost')
print(display_df[sample_cols].head(10).to_string(index=False))

print(f'\n=== Marginal Cost Summary ===')
print(f'  All units     — Min: ${df["marginal_cost"].min():.2f}  Median: ${df["marginal_cost"].median():.2f}  Max: ${df["marginal_cost"].max():.2f}')
thermal_mc = df[df['heat_rate'] > 0]['marginal_cost']
print(f'  Thermal only  — Min: ${thermal_mc.min():.2f}  Median: ${thermal_mc.median():.2f}  Max: ${thermal_mc.max():.2f}')

=== Sample Corrected Marginal Costs (thermal units, sorted) ===
                                    plant_name fuel_category  summer_cap_mw  heat_rate  fuel_price  fuel_cost  vom  carbon_cost  marginal_cost
Princeton University Cogeneration (Mercer, NJ)         Solar            1.2      11.00         0.0        0.0 1.00          0.0           1.00
           WC Landfill Energy LLC (Warren, NJ)         Solar            1.0       3.61         0.0        0.0 1.04          0.0           1.04
Princeton University Cogeneration (Mercer, NJ)         Solar            4.5      11.02         0.0        0.0 1.05          0.0           1.05
                    Rariton OMP (Somerset, NJ)         Solar            2.4       5.37         0.0        0.0 1.05          0.0           1.05
                Yankee Street (Montgomery, OH)         Solar            1.1      23.00         0.0        0.0 1.07          0.0           1.07
          New River Clean Energy (Raleigh, WV)       Biomass            1.6   

## 5. Merit Order Supply Stack

The **merit order** ranks all generating units from cheapest to most expensive. In a competitive wholesale market, units are dispatched in this order to meet demand. The **market clearing price** equals the marginal cost of the last unit needed to serve load.

Construction:
1. Sort all active units by marginal cost (ascending)
2. Compute cumulative capacity
3. Plot as a step function: x = cumulative MW, y = marginal cost $/MWh

In [11]:
# --- Sort by marginal cost, then by capacity (larger units first for ties) ---
df_stack = df.sort_values(['marginal_cost', 'summer_cap_mw'], ascending=[True, False]).copy()
df_stack = df_stack.reset_index(drop=True)

# --- Cumulative capacity ---
df_stack['cum_start'] = df_stack['summer_cap_mw'].cumsum() - df_stack['summer_cap_mw']
df_stack['cum_end']   = df_stack['summer_cap_mw'].cumsum()
df_stack['cum_mid']   = (df_stack['cum_start'] + df_stack['cum_end']) / 2

total_cap = df_stack['summer_cap_mw'].sum()
print(f'Total system capacity in merit order: {total_cap:,.0f} MW')
print(f'Units in stack: {len(df_stack):,}')

# Show where key fuel types start in the merit order
print('\n=== Fuel Type Entry Points (first dispatchable unit by marginal cost) ===')
for fuel in ['Nuclear', 'Wind', 'Solar', 'Hydro', 'Coal', 'Gas CC', 'Gas CT/ST', 'Oil']:
    fuel_df = df_stack[df_stack['fuel_category'] == fuel]
    if len(fuel_df) > 0:
        first = fuel_df.iloc[0]
        print(f'  {fuel:12s}  enters at ${first["marginal_cost"]:6.2f}/MWh  |  cum. {first["cum_start"]:>9,.0f} MW')

Total system capacity in merit order: 217,474 MW
Units in stack: 4,593

=== Fuel Type Entry Points (first dispatchable unit by marginal cost) ===
  Nuclear       enters at $  8.25/MWh  |  cum.    44,167 MW
  Wind          enters at $  2.50/MWh  |  cum.    21,515 MW
  Solar         enters at $  1.00/MWh  |  cum.         0 MW
  Hydro         enters at $  1.12/MWh  |  cum.    13,898 MW
  Coal          enters at $  3.40/MWh  |  cum.    34,011 MW
  Gas CC        enters at $  1.50/MWh  |  cum.    15,732 MW
  Gas CT/ST     enters at $  2.85/MWh  |  cum.    32,000 MW
  Oil           enters at $  3.50/MWh  |  cum.    34,164 MW


In [12]:
# --- Color palette for fuel types ---
fuel_colors = {
    'Nuclear':   '#7b2d8e',
    'Hydro':     '#1f77b4',
    'Wind':      '#2ca02c',
    'Solar':     '#f5c542',
    'Biomass':   '#8c564b',
    'Storage':   '#17becf',
    'Other':     '#aaaaaa',
    'Coal':      '#333333',
    'Gas CC':    '#e36f1e',
    'Gas CT/ST': '#d62728',
    'Oil':       '#9467bd',
}

fuel_order = ['Nuclear', 'Hydro', 'Wind', 'Solar', 'Biomass', 'Storage', 'Other',
              'Coal', 'Gas CC', 'Gas CT/ST', 'Oil']

# --- Build step-function traces per fuel type ---
fig = go.Figure()

for fuel in fuel_order:
    fuel_df = df_stack[df_stack['fuel_category'] == fuel]
    if len(fuel_df) == 0:
        continue

    # Build x, y pairs for step function (horizontal segments)
    x_vals = []
    y_vals = []
    hover_texts = []
    for _, row in fuel_df.iterrows():
        x_vals.extend([row['cum_start'], row['cum_end'], None])
        y_vals.extend([row['marginal_cost'], row['marginal_cost'], None])
        ht = (f"{row['plant_name']}<br>{row['power_hub']}<br>"
              f"Cap: {row['summer_cap_mw']:.0f} MW<br>"
              f"MC: ${row['marginal_cost']:.2f}/MWh<br>"
              f"Fuel Cost: ${row['fuel_cost']:.2f} | VOM: ${row['vom']:.2f}")
        hover_texts.extend([ht, None, None])

    fig.add_trace(go.Scatter(
        x=x_vals, y=y_vals,
        mode='lines',
        name=f"{fuel} ({fuel_df['summer_cap_mw'].sum()/1000:.1f} GW)",
        line=dict(color=fuel_colors.get(fuel, '#999'), width=2.5),
        hovertext=hover_texts,
        hoverinfo='text',
    ))

# --- Typical PJM load range annotation ---
for load, label in [(75000, 'Off-Peak ~75 GW'), (100000, 'Shoulder ~100 GW'),
                     (130000, 'Summer Peak ~130 GW'), (150000, 'Extreme ~150 GW')]:
    fig.add_vline(x=load, line_dash='dot', line_color='gray', opacity=0.5)
    fig.add_annotation(x=load, y=df_stack['marginal_cost'].max() * 0.95,
                       text=label, showarrow=False, textangle=-90,
                       font=dict(size=10, color='gray'))

fig.update_layout(
    title='PJM Merit Order Supply Stack — All Units',
    xaxis_title='Cumulative Capacity (MW)',
    yaxis_title='Marginal Cost ($/MWh)',
    height=600, width=1100,
    legend=dict(title='Fuel Type', x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
    hovermode='closest',
    template='plotly_white',
)
fig.show()

### Zoomed View: Dispatchable Range ($10–$80/MWh)

The interesting price action occurs where thermal units compete. Overnight loads (~70 GW) sit near the coal/gas CC transition zone. Peak loads push into gas CT territory.

In [13]:
# --- Zoomed view: dispatchable thermal range ---
fig_zoom = go.Figure()

for fuel in fuel_order:
    fuel_df = df_stack[df_stack['fuel_category'] == fuel]
    if len(fuel_df) == 0:
        continue

    x_vals = []
    y_vals = []
    hover_texts = []
    for _, row in fuel_df.iterrows():
        x_vals.extend([row['cum_start'], row['cum_end'], None])
        y_vals.extend([row['marginal_cost'], row['marginal_cost'], None])
        ht = (f"{row['plant_name']}<br>{row['power_hub']}<br>"
              f"Cap: {row['summer_cap_mw']:.0f} MW | HR: {row['heat_rate']:.1f}<br>"
              f"MC: ${row['marginal_cost']:.2f}/MWh<br>"
              f"Fuel: ${row['fuel_cost']:.2f} + VOM: ${row['vom']:.2f} + Carbon: ${row['carbon_cost']:.2f}")
        hover_texts.extend([ht, None, None])

    fig_zoom.add_trace(go.Scatter(
        x=x_vals, y=y_vals,
        mode='lines',
        name=fuel,
        line=dict(color=fuel_colors.get(fuel, '#999'), width=3),
        hovertext=hover_texts,
        hoverinfo='text',
    ))

# Load reference lines
for load, label, color in [
    (75000,  'Off-Peak ~75 GW', 'blue'),
    (100000, 'Shoulder ~100 GW', 'green'),
    (130000, 'Summer Peak ~130 GW', 'orange'),
    (150000, 'Extreme ~150 GW', 'red')]:
    fig_zoom.add_vline(x=load, line_dash='dash', line_color=color, opacity=0.6)
    fig_zoom.add_annotation(x=load, y=78, text=label, showarrow=False,
                            textangle=-90, font=dict(size=10, color=color))

fig_zoom.update_layout(
    title='PJM Merit Order — Thermal Dispatch Range',
    xaxis_title='Cumulative Capacity (MW)',
    yaxis_title='Marginal Cost ($/MWh)',
    xaxis_range=[60000, 180000],
    yaxis_range=[10, 80],
    height=600, width=1100,
    legend=dict(title='Fuel Type'),
    hovermode='closest',
    template='plotly_white',
)
fig_zoom.show()

## 6. Capacity & Cost Summary by Fuel Type

In [14]:
# --- Capacity summary by fuel type ---
summary = df_stack.groupby('fuel_category').agg(
    unit_count=('plant_name', 'count'),
    total_mw=('summer_cap_mw', 'sum'),
    avg_mc=('marginal_cost', 'mean'),
    min_mc=('marginal_cost', 'min'),
    max_mc=('marginal_cost', 'max'),
    avg_hr=('heat_rate', lambda x: x[x > 0].mean() if (x > 0).any() else 0),
    avg_co2=('co2_rate', lambda x: x[x > 0].mean() if (x > 0).any() else 0),
).round(2)

summary['pct_cap'] = (summary['total_mw'] / summary['total_mw'].sum() * 100).round(1)
summary = summary.sort_values('avg_mc')
summary['total_mw'] = summary['total_mw'].apply(lambda x: f'{x:,.0f}')

print('=== PJM Capacity & Cost Summary by Fuel Type ===')
print(summary[['unit_count', 'total_mw', 'pct_cap', 'avg_mc', 'min_mc', 'max_mc',
               'avg_hr', 'avg_co2']].to_string())

=== PJM Capacity & Cost Summary by Fuel Type ===
               unit_count total_mw  pct_cap  avg_mc  min_mc  max_mc  avg_hr  avg_co2
fuel_category                                                                       
Solar                1644   15,522      7.1    1.03    1.00    1.12   10.80     0.00
Wind                  146   11,122      5.1    2.70    2.50    5.00    0.00     0.00
Biomass               652    2,004      0.9    2.96    2.00    9.25   13.28     0.00
Other                   5      113      0.1    4.27    3.50    6.00    5.32     0.00
Storage                92      686      0.3    5.03    5.00    5.40    0.00     0.00
Hydro                 308    8,375      3.9    8.01    1.12   10.42    0.00     0.00
Nuclear                31   32,672     15.0    8.45    8.25    8.66    9.57     0.00
Gas CC                313   59,512     27.4   26.89    1.50   69.09    8.48     0.45
Gas CT/ST             684   36,308     16.7   31.83    2.85   96.56   12.10     0.64
Coal            

In [15]:
# --- Capacity mix by fuel type ---
cap_summary = df_stack.groupby('fuel_category')['summer_cap_mw'].sum().sort_values(ascending=True)

fig_cap = px.bar(
    x=cap_summary.values,
    y=cap_summary.index,
    orientation='h',
    color=cap_summary.index,
    color_discrete_map=fuel_colors,
    labels={'x': 'Installed Capacity (MW)', 'y': 'Fuel Type'},
    title='PJM Installed Capacity by Fuel Type',
)
fig_cap.update_layout(
    height=450, width=800,
    showlegend=False,
    template='plotly_white',
    xaxis_tickformat=',',
)
# Add MW labels
for i, (val, fuel) in enumerate(zip(cap_summary.values, cap_summary.index)):
    fig_cap.add_annotation(x=val, y=fuel, text=f' {val/1000:.1f} GW',
                           showarrow=False, xanchor='left', font=dict(size=11))
fig_cap.show()

In [16]:
# --- Marginal cost distribution by fuel type (thermal only) ---
thermal_df = df_stack[df_stack['fuel_category'].isin(['Coal', 'Gas CC', 'Gas CT/ST', 'Oil'])]

fig_box = px.box(
    thermal_df,
    x='fuel_category',
    y='marginal_cost',
    color='fuel_category',
    color_discrete_map=fuel_colors,
    category_orders={'fuel_category': ['Coal', 'Gas CC', 'Gas CT/ST', 'Oil']},
    labels={'marginal_cost': 'Marginal Cost ($/MWh)', 'fuel_category': 'Fuel Type'},
    title='Marginal Cost Distribution — Thermal Units',
    points='outliers',
)
fig_box.update_layout(height=500, width=700, showlegend=False, template='plotly_white')
fig_box.show()

### Implied Clearing Prices at Key Load Levels

Using the merit order, we can read off the system marginal cost (clearing price) at different demand levels.

In [17]:
# --- Clearing price at various load levels ---
load_levels = [65000, 75000, 85000, 95000, 100000, 110000, 120000, 130000, 140000, 150000]

print(f'{"Load (MW)":>12s}  {"Clearing Price":>15s}  {"Marginal Fuel":>15s}  {"Marginal Unit":<40s}')
print('-' * 90)
for load in load_levels:
    # Find the unit that pushes cumulative capacity past the load level
    marginal = df_stack[df_stack['cum_end'] >= load]
    if len(marginal) > 0:
        unit = marginal.iloc[0]
        print(f'{load:>12,}  ${unit["marginal_cost"]:>13.2f}  {unit["fuel_category"]:>15s}  {str(unit["plant_name"]):<40s}')
    else:
        print(f'{load:>12,}  {"EXCEEDS CAPACITY":>15s}')

   Load (MW)   Clearing Price    Marginal Fuel  Marginal Unit                           
------------------------------------------------------------------------------------------
      65,000  $         8.48          Nuclear  Donald C Cook (Berrien, MI)             
      75,000  $         8.63          Nuclear  Dresden Generating Station (Grundy, IL) 
      85,000  $        18.48           Gas CC  Lackawanna Energy Center (Lackawanna, PA)
      95,000  $        19.94           Gas CC  CPV Fairview Energy Center (Cambria, PA)
     100,000  $        20.79           Gas CC  Nelson Energy Center (Lee, IL)          
     110,000  $        22.99           Gas CC  Ironwood LLC (Lebanon, PA)              
     120,000  $        25.09             Coal  Powerton (Tazewell, IL)                 
     130,000  $        28.21           Gas CC  PSEG Linden Generating Station (Union, NJ)
     140,000  $        30.22           Gas CC  Fremont Energy Center (Sandusky, OH)    
     150,000  $        31

## 7. Marginal Abatement Cost Curve (MACC)

### Approach

A MACC shows the **cost to reduce CO2 emissions** by switching from dirtier to cleaner generation. For each fossil unit, we compare it against a **reference clean technology** — the most efficient gas combined-cycle (CC) unit — and calculate:

| Variable | Formula |
|----------|---------|
| **Abatement rate** | CO2_unit − CO2_reference (tons CO2/MWh) |
| **Cost difference** | MC_reference − MC_unit ($/MWh) |
| **Abatement cost** | Cost_diff / Abatement_rate ($/ton CO2) |
| **Abatement potential** | Abatement_rate × Capacity (tons CO2/hr) |

**Interpretation:**
- **Negative abatement cost** → switching to gas CC is *cheaper* AND *cleaner* (win-win)
- **Positive abatement cost** → switching is cleaner but more expensive
- Units are sorted left to right by abatement cost

### Limitations

This MACC captures **variable cost fuel-switching** only. It does not include:
- Capital costs of building new gas CC capacity
- Transmission constraints limiting actual dispatch flexibility
- Reliability/ancillary service requirements
- Start-up costs and minimum run times

In [18]:
# --- Filter to fossil units with positive heat rate ---
fossil = df_stack[
    (df_stack['fuel_base'].isin(['Gas', 'Coal', 'Oil'])) &
    (df_stack['heat_rate'] > 0) &
    (df_stack['summer_cap_mw'] > 0)
].copy()

print(f'Fossil units with HR > 0: {len(fossil):,}  |  Capacity: {fossil["summer_cap_mw"].sum():,.0f} MW')

# --- CO2 intensity summary by fuel type ---
co2_summary = fossil.groupby('fuel_category').agg(
    count=('co2_rate', 'count'),
    total_mw=('summer_cap_mw', 'sum'),
    avg_co2_rate=('co2_rate', 'mean'),
    avg_mc=('marginal_cost', 'mean'),
    avg_hr=('heat_rate', 'mean'),
).round(3)
print('\n=== CO2 Intensity by Fuel Type ===')
print(co2_summary.to_string())

Fossil units with HR > 0: 1,299  |  Capacity: 139,734 MW

=== CO2 Intensity by Fuel Type ===
               count   total_mw  avg_co2_rate   avg_mc  avg_hr
fuel_category                                                 
Coal             125  46031.570         1.098   36.134  11.316
Gas CC           302  54177.092         0.449   27.818   8.478
Gas CT/ST        531  35348.100         0.641   39.619  12.099
Oil              341   4177.348         0.986  221.249  13.140


In [19]:
# --- Emissions merit order: CO2 intensity vs cumulative capacity ---
fossil_sorted = fossil.sort_values('co2_rate', ascending=True).copy()
fossil_sorted['cum_start_f'] = fossil_sorted['summer_cap_mw'].cumsum() - fossil_sorted['summer_cap_mw']
fossil_sorted['cum_end_f'] = fossil_sorted['summer_cap_mw'].cumsum()

fig_co2 = go.Figure()

for fuel in ['Gas CC', 'Gas CT/ST', 'Coal', 'Oil']:
    fuel_df = fossil_sorted[fossil_sorted['fuel_category'] == fuel]
    if len(fuel_df) == 0:
        continue
    x_vals, y_vals = [], []
    for _, row in fuel_df.iterrows():
        x_vals.extend([row['cum_start_f'], row['cum_end_f'], None])
        y_vals.extend([row['co2_rate'], row['co2_rate'], None])
    fig_co2.add_trace(go.Scatter(
        x=x_vals, y=y_vals,
        mode='lines', name=fuel,
        line=dict(color=fuel_colors.get(fuel, '#999'), width=3),
    ))

fig_co2.update_layout(
    title='PJM Emissions Merit Order — CO₂ Intensity by Unit',
    xaxis_title='Cumulative Fossil Capacity (MW)',
    yaxis_title='CO₂ Emission Rate (tons CO₂/MWh)',
    height=500, width=1000,
    template='plotly_white',
    legend=dict(title='Fuel Type'),
)
fig_co2.show()

In [20]:
# --- Define reference technology: most efficient gas CC ---
gas_cc = fossil[fossil['fuel_category'] == 'Gas CC'].copy()
ref_hr = gas_cc['heat_rate'].quantile(0.10)          # 10th percentile = most efficient
ref_fuel_price = gas_cc['fuel_price'].median()
ref_fuel_cost = ref_hr * ref_fuel_price
ref_vom = gas_cc['vom'].quantile(0.10)
ref_co2_rate = ref_hr * ef_gas
ref_mc = ref_fuel_cost + ref_vom

print(f'=== Reference Technology: Efficient Gas CC (10th pctile HR) ===')
print(f'  Heat Rate:     {ref_hr:.2f} MMBTU/MWh')
print(f'  Fuel Price:    ${ref_fuel_price:.2f}/MMBtu')
print(f'  Fuel Cost:     ${ref_fuel_cost:.2f}/MWh')
print(f'  VOM:           ${ref_vom:.2f}/MWh')
print(f'  Marginal Cost: ${ref_mc:.2f}/MWh')
print(f'  CO2 Rate:      {ref_co2_rate:.4f} tons/MWh')

# --- Calculate abatement for units dirtier than the reference ---
macc = fossil[fossil['co2_rate'] > ref_co2_rate].copy()

macc['abatement_rate'] = macc['co2_rate'] - ref_co2_rate    # tons CO2/MWh abated
macc['cost_diff'] = ref_mc - macc['marginal_cost']           # $/MWh (neg = ref is cheaper)
macc['abatement_cost'] = macc['cost_diff'] / macc['abatement_rate']  # $/ton CO2

# Total abatement potential = rate x capacity (tons CO2/hr)
macc['abatement_potential'] = macc['abatement_rate'] * macc['summer_cap_mw']

# Sort by abatement cost
macc = macc.sort_values('abatement_cost', ascending=True).reset_index(drop=True)
macc['cum_abatement'] = macc['abatement_potential'].cumsum()
macc['cum_abatement_start'] = macc['cum_abatement'] - macc['abatement_potential']

total_abatement = macc['abatement_potential'].sum()
negative_abatement = macc[macc['abatement_cost'] < 0]['abatement_potential'].sum()

print(f'\n=== MACC Summary ===')
print(f'  Units in MACC: {len(macc):,}')
print(f'  Total abatement potential: {total_abatement:,.0f} tons CO2/hr')
print(f'  Negative cost (win-win): {negative_abatement:,.0f} tons CO2/hr ({negative_abatement/total_abatement*100:.1f}%)')
print(f'  Abatement cost range: ${macc["abatement_cost"].min():.2f} to ${macc["abatement_cost"].max():.2f} /ton CO2')

=== Reference Technology: Efficient Gas CC (10th pctile HR) ===
  Heat Rate:     6.73 MMBTU/MWh
  Fuel Price:    $2.65/MMBtu
  Fuel Cost:     $17.83/MWh
  VOM:           $1.50/MWh
  Marginal Cost: $19.33/MWh
  CO2 Rate:      0.3567 tons/MWh

=== MACC Summary ===
  Units in MACC: 1,237
  Total abatement potential: 45,810 tons CO2/hr
  Negative cost (win-win): 44,197 tons CO2/hr (96.5%)
  Abatement cost range: $-3945.41 to $184.65 /ton CO2


In [21]:
# --- MACC Bar Chart ---
fig_macc = go.Figure()

for fuel in ['Coal', 'Gas CT/ST', 'Oil']:
    fuel_df = macc[macc['fuel_category'] == fuel]
    if len(fuel_df) == 0:
        continue

    hover_texts = []
    for _, r in fuel_df.iterrows():
        hover_texts.append(
            f"{r['plant_name']}<br>{r['power_hub']}<br>"
            f"Cap: {r['summer_cap_mw']:.0f} MW<br>"
            f"CO2: {r['co2_rate']:.3f} t/MWh -> {ref_co2_rate:.3f} t/MWh<br>"
            f"MC: ${r['marginal_cost']:.2f} -> ${ref_mc:.2f}/MWh<br>"
            f"Abatement cost: ${r['abatement_cost']:.2f}/ton CO2"
        )

    fig_macc.add_trace(go.Bar(
        x=(fuel_df['cum_abatement_start'] + fuel_df['cum_abatement']) / 2,
        y=fuel_df['abatement_cost'],
        width=fuel_df['abatement_potential'],
        name=fuel,
        marker_color=fuel_colors.get(fuel, '#999'),
        hovertext=hover_texts,
        hoverinfo='text',
    ))

# Zero line
fig_macc.add_hline(y=0, line_dash='solid', line_color='black', line_width=1)

# RGGI price reference
fig_macc.add_hline(y=rggi_price, line_dash='dash', line_color='green', opacity=0.7,
                   annotation_text=f'RGGI Price: ${rggi_price}/ton',
                   annotation_position='top right')

fig_macc.update_layout(
    title='PJM Marginal Abatement Cost Curve — Fuel Switching to Efficient Gas CC',
    xaxis_title='Cumulative Abatement Potential (tons CO2/hr)',
    yaxis_title='Abatement Cost ($/ton CO2)',
    height=600, width=1100,
    barmode='overlay',
    template='plotly_white',
    legend=dict(title='Displaced Fuel'),
)

# Add annotation for negative cost region
if negative_abatement > 0:
    fig_macc.add_annotation(
        x=negative_abatement / 2,
        y=macc[macc['abatement_cost'] < 0]['abatement_cost'].median(),
        text='Win-Win Zone<br>(Cheaper AND Cleaner)',
        showarrow=True, arrowhead=2,
        font=dict(size=12, color='green'),
    )

fig_macc.show()

In [22]:
# --- MACC summary by fuel type ---
print('=== Abatement Cost Summary by Fuel Type (switching to efficient gas CC) ===\n')
for fuel in ['Coal', 'Gas CT/ST', 'Oil']:
    fuel_df = macc[macc['fuel_category'] == fuel]
    if len(fuel_df) == 0:
        continue
    neg = fuel_df[fuel_df['abatement_cost'] < 0]
    pos = fuel_df[fuel_df['abatement_cost'] >= 0]
    print(f'  {fuel}:')
    print(f'    Units: {len(fuel_df):,}  |  Capacity: {fuel_df["summer_cap_mw"].sum():,.0f} MW')
    print(f'    Avg CO2 rate: {fuel_df["co2_rate"].mean():.3f} tons/MWh')
    print(f'    Avg marginal cost: ${fuel_df["marginal_cost"].mean():.2f}/MWh')
    print(f'    Abatement cost: ${fuel_df["abatement_cost"].mean():.2f}/ton CO2 (avg)')
    print(f'    Win-win (neg cost): {len(neg)} units, {neg["summer_cap_mw"].sum():,.0f} MW')
    print(f'    Positive cost:      {len(pos)} units, {pos["summer_cap_mw"].sum():,.0f} MW')
    print()

=== Abatement Cost Summary by Fuel Type (switching to efficient gas CC) ===

  Coal:
    Units: 125  |  Capacity: 46,032 MW
    Avg CO2 rate: 1.098 tons/MWh
    Avg marginal cost: $36.13/MWh
    Abatement cost: $-22.18/ton CO2 (avg)
    Win-win (neg cost): 118 units, 43,712 MW
    Positive cost:      7 units, 2,320 MW

  Gas CT/ST:
    Units: 510  |  Capacity: 34,392 MW
    Avg CO2 rate: 0.659 tons/MWh
    Avg marginal cost: $40.54/MWh
    Abatement cost: $-80.93/ton CO2 (avg)
    Win-win (neg cost): 510 units, 34,392 MW
    Positive cost:      0 units, 0 MW

  Oil:
    Units: 333  |  Capacity: 4,123 MW
    Avg CO2 rate: 1.004 tons/MWh
    Avg marginal cost: $225.25/MWh
    Abatement cost: $-368.00/ton CO2 (avg)
    Win-win (neg cost): 333 units, 4,123 MW
    Positive cost:      0 units, 0 MW



### CO2 Intensity vs. Marginal Cost Scatter

Each dot is a fossil generating unit. Size = capacity. Units in the upper-right are both expensive and dirty — prime candidates for displacement.

In [23]:
fig_scatter = px.scatter(
    fossil,
    x='co2_rate',
    y='marginal_cost',
    color='fuel_category',
    size='summer_cap_mw',
    size_max=30,
    color_discrete_map=fuel_colors,
    hover_data={'plant_name': True, 'power_hub': True, 'heat_rate': ':.1f',
                'summer_cap_mw': ':,.0f', 'co2_rate': ':.3f', 'marginal_cost': ':$.2f'},
    labels={
        'co2_rate': 'CO2 Emission Rate (tons/MWh)',
        'marginal_cost': 'Marginal Cost ($/MWh)',
        'fuel_category': 'Fuel Type',
        'summer_cap_mw': 'Capacity (MW)',
    },
    title='PJM Fossil Units — CO2 Intensity vs. Marginal Cost',
)
# Add reference point
fig_scatter.add_trace(go.Scatter(
    x=[ref_co2_rate], y=[ref_mc],
    mode='markers+text',
    marker=dict(size=15, color='green', symbol='star'),
    text=['Reference Gas CC'],
    textposition='top right',
    showlegend=False,
))
fig_scatter.update_layout(
    height=600, width=1000,
    template='plotly_white',
)
fig_scatter.show()

## 8. Key Findings

### Merit Order
1. **Must-run capacity (~78 GW)** — nuclear, hydro, wind, solar, biomass — forms the base of the stack at near-zero marginal cost ($0–6/MWh, VOM only).
2. **Coal units** enter the stack around $15–35/MWh depending on coal basin and heat rate.
3. **Gas CC units** are competitive with coal at current gas prices (~$2.60/MMBtu), entering at $18–28/MWh.
4. **Gas CT/peakers** fill the upper portion at $30–45/MWh, becoming marginal during high-load hours.
5. **Oil units** are the most expensive at $100–280/MWh, dispatched only during extreme peaks.
6. The gas-coal **crossover** is very tight at current gas prices — small movements in Henry Hub can flip the dispatch order.

### MACC
1. At current PJM gas prices, **most fuel-switching from coal/CT/oil to efficient gas CC has negative abatement cost** — it is simultaneously cheaper AND cleaner.
2. This explains the ongoing **coal-to-gas switching** trend in PJM, driven by economics rather than carbon policy.
3. The RGGI carbon price ($15/ton CO2) adds ~$5–8/MWh to coal unit costs in participating states, further accelerating the switch.
4. Oil-fired generation has extremely high abatement cost savings (highly negative $/ton) but limited abatement volume since oil rarely runs.
5. **Remaining positive-cost abatement** (if any) represents units where gas CC displacement would be cleaner but marginally more expensive — these are the units where carbon pricing has the most impact.

### Limitations
- This analysis uses **variable costs only** (no capital, start-up, or fixed costs).
- The MACC assumes sufficient gas CC capacity exists to displace other fuels — in practice, transmission and siting constraints limit this.
- Heat rates and VOM from the workbook are static; actual values vary with load level, ambient temperature, and unit age.